# Socratic Debug Tutor — QLoRA training on a free T4

Runs the training phases that need a CUDA GPU. Everything else (scenarios,
evaluation harness, teacher generation, quality gate) runs on CPU in the repo.

**Before you start**

1. Runtime → Change runtime type → **T4 GPU**.
2. Have `data/accepted/v1.jsonl` ready — produced by the quality gate on your
   own machine. This notebook does not generate data; generation is an API
   step, not a GPU step.
3. An `ANTHROPIC_API_KEY` is needed **only** for the evaluation cells at the
   end (the LLM judge). Training itself needs no credentials.

**What this produces**

- `outputs/socratic-v1/` — the main adapter
- `outputs/socratic-n{125,250,500,...}/` — the data-efficiency sweep
- `results/base_vs_tuned/` and `results/data_efficiency/` if you run the
  evaluation cells here rather than back on your own machine

## 1. Confirm the GPU can do 4-bit

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# bitsandbytes NF4 needs compute capability >= 7.5.
# T4 = 7.5 (ok), L4 = 8.9 (ok), A100 = 8.0 (ok), P100/K80 = too old.

## 2. Get the repository

In [ ]:
# Option A — clone from your remote:
# !git clone https://github.com/<you>/socratic-debug-tutor.git
# %cd socratic-debug-tutor

# Option B — upload a zip of the repo, then:
# from google.colab import files; files.upload()
# !unzip -q socratic-debug-tutor.zip && cd socratic-debug-tutor

import os
print(os.getcwd())
!ls

## 3. Install the training stack

Colab ships a CUDA torch already, so only the fine-tuning libraries are added.

In [ ]:
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9" \
    "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20" \
    "pydantic>=2.7" pyyaml python-dotenv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name} | cc {p.major}.{p.minor} | {p.total_memory/2**30:.1f} GiB')

## 4. Supply the accepted dataset

Upload the `data/accepted/v1.jsonl` produced by `scripts/filter_data.py`.

In [ ]:
from pathlib import Path
Path('data/accepted').mkdir(parents=True, exist_ok=True)

# from google.colab import files
# uploaded = files.upload()   # choose v1.jsonl
# Path('data/accepted/v1.jsonl').write_bytes(next(iter(uploaded.values())))

!wc -l data/accepted/v1.jsonl

## 5. Configure for the T4

A T4 has no bfloat16, so compute dtype and mixed precision both move to fp16.

In [ ]:
import yaml
from pathlib import Path

path = Path('training/configs/qlora_qwen3_1_7b.yaml')
cfg = yaml.safe_load(path.read_text())

cfg['quantization']['bnb_4bit_compute_dtype'] = 'float16'
cfg['training']['bf16'] = False
cfg['training']['fp16'] = True

# Pin the base-model revision before the final run so results are reproducible.
# from huggingface_hub import HfApi
# cfg['model']['revision'] = HfApi().model_info('Qwen/Qwen3-1.7B').sha

path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

## 6. Validate before spending GPU time

The dry run builds the dataset, checks train/eval contamination and writes the
checkpoint metadata without loading a model.

In [ ]:
!python -m training.train --dry-run --run-name socratic-v1

## 7. Train the main adapter

In [ ]:
!python -m training.train --run-name socratic-v1

## 8. Data-efficiency sweep

Four checkpoints on **nested** subsets, everything except N held constant.
Each run is short; the whole sweep on a T4 is roughly 1–3 hours for ~600
examples. `--plan` first to see the sizes that fit your dataset.

In [ ]:
!python -m ablations.data_efficiency --plan

In [ ]:
!python -m ablations.data_efficiency --train

## 9. Evaluate (needs the judge credential)

Both models see the **same** weak `zero_shot` prompt, the same held-out
scenarios and the same judge. Only the adapter differs.

In [ ]:
import os
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!pip install -q anthropic

!python -m ablations.base_vs_tuned \
    --base hf:Qwen/Qwen3-1.7B \
    --tuned 'peft:Qwen/Qwen3-1.7B+outputs/socratic-v1' \
    --judge anthropic:claude-opus-5

In [ ]:
!python -m ablations.data_efficiency --evaluate --judge anthropic:claude-opus-5

## 10. Take the artifacts home

In [ ]:
!python scripts/build_manifest.py
!tar -czf socratic-artifacts.tar.gz outputs results

# from google.colab import files; files.download('socratic-artifacts.tar.gz')

# Or publish the adapter (small — a few tens of MB):
# from huggingface_hub import login, HfApi
# login(token=userdata.get('HF_TOKEN'))
# HfApi().upload_folder(folder_path='outputs/socratic-v1',
#                       repo_id='<you>/socratic-debug-tutor-qwen3-1.7b',
#                       repo_type='model')

## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `CUDA out of memory` | batch or sequence too large | drop `per_device_train_batch_size` to 1 and raise `gradient_accumulation_steps` to 16 |
| `Compute capability < 7.5` | pre-Turing GPU | switch runtime to T4 |
| `bf16 is not supported` | T4 has no bfloat16 | re-run cell 5 |
| `ContaminationError` | training data overlaps the eval set | re-run the quality gate, which drops contaminated examples |
| `No accepted examples` | dataset not uploaded | re-run cell 4 |